# Customer Churn Analysis

## Business Objective
Build a customer churn prediction model that identifies customers at higher risk of churn and provides a 0-100 churn score for retention actions.

## 1. Import Libraries

In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix

## 2. Load Data

In [28]:
df = pd.read_csv('../output/churn_model_data.csv')
print('Shape:', df.shape)
df.head()

Shape: (200, 13)


,Total_Units,Total_Net_Revenue,Average_Transaction_Value,Average_Discount,Return_Rate,Unique_Channels,Customer_City,Age_Band,Gender,Acquisition_Channel,Total_Orders,Total_Revenue,Churned
0,14,27005.80,5401.16,6.0,20.0,4,Delhi,18-24,F,Marketplace,19,33219,0
1,16,41664.80,8332.96,1.0,0.0,2,Chennai,45-54,F,App,3,35834,0
2,19,29612.45,5922.49,5.0,20.0,3,Bangalore,35-44,F,App,19,5153,0
3,15,29879.35,5975.87,6.0,20.0,2,Delhi,55+,M,Website,19,10568,0
4,17,52209.80,10441.96,3.0,0.0,3,Delhi,45-54,F,Marketplace,8,20359,0


## 3. Prepare Features and Target

In [29]:
X = df.drop(columns=['Churned'])
y = df['Churned']
print('X shape:', X.shape)
print('Target distribution:')
print(y.value_counts())

X shape: (200, 12)
Target distribution:
Churned
0    194
1      6
Name: count, dtype: int64


## 4. Train-Test Split

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

Training rows: 160
Testing rows: 40


## 5. Build Logistic Regression Model

In [31]:
categorical_features = ['Customer_City', 'Age_Band', 'Gender', 'Acquisition_Channel']
numeric_features = [c for c in X.columns if c not in categorical_features]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
print('Model training completed.')

Model training completed.


## 6. Model Evaluation

In [32]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print('AUC:', round(auc, 4))
print('Accuracy:', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall:', round(recall, 4))
print('Confusion Matrix:')
print(cm)

AUC: 0.9744
Accuracy: 0.975
Precision: 0.5
Recall: 1.0
Confusion Matrix:
[[38  1]
 [ 0  1]]


## 7. Feature Importance

Positive logistic-regression coefficients increase predicted churn probability, while negative coefficients decrease it, holding other features constant. These are model associations, not causal effects.

In [33]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

importance['Absolute_Coefficient'] = importance['Coefficient'].abs()
importance = importance.sort_values('Absolute_Coefficient', ascending=False)
importance.head(10)

,Feature,Coefficient,Absolute_Coefficient
23,num__Total_Orders,-2.839239,2.839239
0,cat__Customer_City_Bangalore,1.164978,1.164978
1,cat__Customer_City_Chennai,-1.107862,1.107862
5,cat__Customer_City_Pune,1.026403,1.026403
9,cat__Age_Band_45-54,-0.840833,0.840833
17,num__Total_Units,0.797178,0.797178
8,cat__Age_Band_35-44,0.689590,0.689590
3,cat__Customer_City_Hyderabad,-0.672018,0.672018
14,cat__Acquisition_Channel_Marketplace,-0.643707,0.643707
6,cat__Age_Band_18-24,-0.640657,0.640657


## 8. Customer Churn Scores

In [34]:
customer_ids = pd.read_csv('../output/customer_features.csv')['Customer_ID']
all_prob = model.predict_proba(X)[:, 1]

scores = pd.DataFrame({
    'Customer_ID': customer_ids,
    'Churned': y,
    'Churn_Probability': all_prob,
    'Churn_Score': (all_prob * 100).round(2)
})

scores['Risk_Category'] = pd.cut(
    scores['Churn_Score'],
    bins=[-1, 39.99, 69.99, 100],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

scores = scores.sort_values('Churn_Score', ascending=False)
scores.head(10)

,Customer_ID,Churned,Churn_Probability,Churn_Score,Risk_Category
176,C78,1,0.967577,96.76,High Risk
199,C99,1,0.950341,95.03,High Risk
101,C190,1,0.946850,94.68,High Risk
149,C53,1,0.925295,92.53,High Risk
51,C145,1,0.920718,92.07,High Risk
164,C67,0,0.900176,90.02,High Risk
139,C44,0,0.775648,77.56,High Risk
121,C28,0,0.591459,59.15,Medium Risk
67,C16,1,0.526175,52.62,Medium Risk
37,C132,0,0.483622,48.36,Medium Risk


## 9. Score-60 Threshold Analysis

In [35]:
threshold = 60
threshold_pred = (scores['Churn_Score'] >= threshold).astype(int)

tp = ((scores['Churned'] == 1) & (threshold_pred == 1)).sum()
fp = ((scores['Churned'] == 0) & (threshold_pred == 1)).sum()
tn = ((scores['Churned'] == 0) & (threshold_pred == 0)).sum()
fn = ((scores['Churned'] == 1) & (threshold_pred == 0)).sum()

fpr = fp / (fp + tn) * 100
recall_60 = tp / (tp + fn) * 100

print('Customers flagged:', int(threshold_pred.sum()))
print('True Positives:', int(tp))
print('False Positives:', int(fp))
print('True Negatives:', int(tn))
print('False Negatives:', int(fn))
print('False Positive Rate:', round(fpr, 2), '%')
print('Recall:', round(recall_60, 2), '%')
print('Acceptance Criterion:', 'PASS' if fpr < 25 else 'FAIL')

Customers flagged: 7
True Positives: 5
False Positives: 2
True Negatives: 192
False Negatives: 1
False Positive Rate: 1.03 %
Recall: 83.33 %
Acceptance Criterion: PASS


## 10. Business Interpretation

The model assigns each customer a churn probability and a 0-100 churn score. Customers with scores of 70 or above are classified as High Risk, 40-69.99 as Medium Risk, and below 40 as Low Risk.

At a score threshold of 60, the analysis identified 5 of 6 churned customers, with 2 false positives. The resulting false-positive rate was 1.03% and recall was 83.33%.

Because only 6 customers are labeled as churned, these metrics should be interpreted cautiously and validated on a larger, more representative dataset before production use.

## 11. Output Files

- `output/customer_features.csv` — engineered customer-level features
- `output/churn_model_data.csv` — model-ready dataset
- `output/customer_churn_scores.csv` — customer churn scores and risk categories
- `output/model_summary.txt` — model evaluation summary

In [36]:
df = pd.read_csv('../output/churn_model_data.csv')

print(df.head())
print("Shape:", df.shape)

   Total_Units  Total_Net_Revenue  Average_Transaction_Value  \
0           14           27005.80                    5401.16   
1           16           41664.80                    8332.96   
2           19           29612.45                    5922.49   
3           15           29879.35                    5975.87   
4           17           52209.80                   10441.96   

   Average_Discount  Return_Rate  Unique_Channels Customer_City Age_Band  \
0               6.0         20.0                4         Delhi    18-24   
1               1.0          0.0                2       Chennai    45-54   
2               5.0         20.0                3     Bangalore    35-44   
3               6.0         20.0                2         Delhi      55+   
4               3.0          0.0                3         Delhi    45-54   

  Gender Acquisition_Channel  Total_Orders  Total_Revenue  Churned  
0      F         Marketplace            19          33219        0  
1      F            

In [37]:
X = df.drop(columns=['Churned'])
y = df['Churned']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nChurn distribution:")
print(y.value_counts())

Features shape: (200, 12)
Target shape: (200,)

Churn distribution:
Churned
0    194
1      6
Name: count, dtype: int64


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining churn distribution:")
print(y_train.value_counts())

print("\nTesting churn distribution:")
print(y_test.value_counts())

X_train: (160, 12)
X_test: (40, 12)
y_train: (160,)
y_test: (40,)

Training churn distribution:
Churned
0    155
1      5
Name: count, dtype: int64

Testing churn distribution:
Churned
0    39
1     1
Name: count, dtype: int64


In [39]:
categorical_features = [
    'Customer_City',
    'Age_Band',
    'Gender',
    'Acquisition_Channel'
]

numeric_features = [
    'Total_Units',
    'Total_Net_Revenue',
    'Average_Transaction_Value',
    'Average_Discount',
    'Return_Rate',
    'Unique_Channels',
    'Total_Orders',
    'Total_Revenue'
]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: ['Customer_City', 'Age_Band', 'Gender', 'Acquisition_Channel']
Numeric features: ['Total_Units', 'Total_Net_Revenue', 'Average_Transaction_Value', 'Average_Discount', 'Return_Rate', 'Unique_Channels', 'Total_Orders', 'Total_Revenue']


In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=42
        ))
    ]
)

model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [41]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("AUC:", round(auc, 4))
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))

print("\nConfusion Matrix:")
print(cm)

AUC: 0.9744
Accuracy: 0.975
Precision: 0.5
Recall: 1.0

Confusion Matrix:
[[38  1]
 [ 0  1]]


In [42]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

feature_importance['Absolute_Importance'] = (
    feature_importance['Coefficient'].abs()
)

feature_importance = feature_importance.sort_values(
    'Absolute_Importance',
    ascending=False
)

print(feature_importance.head(10).to_string(index=False))

                             Feature  Coefficient  Absolute_Importance
                   num__Total_Orders    -2.839239             2.839239
        cat__Customer_City_Bangalore     1.164978             1.164978
          cat__Customer_City_Chennai    -1.107862             1.107862
             cat__Customer_City_Pune     1.026403             1.026403
                 cat__Age_Band_45-54    -0.840833             0.840833
                    num__Total_Units     0.797178             0.797178
                 cat__Age_Band_35-44     0.689590             0.689590
        cat__Customer_City_Hyderabad    -0.672018             0.672018
cat__Acquisition_Channel_Marketplace    -0.643707             0.643707
                 cat__Age_Band_18-24    -0.640657             0.640657


In [43]:
# Get churn probabilities for all customers
all_probabilities = model.predict_proba(X)[:, 1]

# Load original customer data to get Customer_ID
original_df = pd.read_csv('../output/customer_features.csv')

customer_scores = original_df[['Customer_ID', 'Churned']].copy()

customer_scores['Churn_Probability'] = all_probabilities
customer_scores['Churn_Score'] = (
    customer_scores['Churn_Probability'] * 100
).round(2)

customer_scores = customer_scores.sort_values(
    'Churn_Score',
    ascending=False
)

print(customer_scores.head(10).to_string(index=False))

Customer_ID  Churned  Churn_Probability  Churn_Score
        C78        1           0.967577        96.76
        C99        1           0.950341        95.03
       C190        1           0.946850        94.68
        C53        1           0.925295        92.53
       C145        1           0.920718        92.07
        C67        0           0.900176        90.02
        C44        0           0.775648        77.56
        C28        0           0.591459        59.15
        C16        1           0.526175        52.62
       C132        0           0.483622        48.36


In [44]:
def risk_category(score):
    if score >= 70:
        return 'High Risk'
    elif score >= 40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

customer_scores['Risk_Category'] = (
    customer_scores['Churn_Score'].apply(risk_category)
)

print(customer_scores.head(10).to_string(index=False))

print("\nRisk Category Counts:")
print(customer_scores['Risk_Category'].value_counts())

Customer_ID  Churned  Churn_Probability  Churn_Score Risk_Category
        C78        1           0.967577        96.76     High Risk
        C99        1           0.950341        95.03     High Risk
       C190        1           0.946850        94.68     High Risk
        C53        1           0.925295        92.53     High Risk
       C145        1           0.920718        92.07     High Risk
        C67        0           0.900176        90.02     High Risk
        C44        0           0.775648        77.56     High Risk
        C28        0           0.591459        59.15   Medium Risk
        C16        1           0.526175        52.62   Medium Risk
       C132        0           0.483622        48.36   Medium Risk

Risk Category Counts:
Risk_Category
Low Risk       188
High Risk        7
Medium Risk      5
Name: count, dtype: int64


In [45]:
output_file = '../output/customer_churn_scores.csv'

customer_scores.to_csv(output_file, index=False)

print("File saved successfully!")
print("Saved to:", output_file)
print("Rows:", len(customer_scores))
print("Columns:", len(customer_scores.columns))

File saved successfully!
Saved to: ../output/customer_churn_scores.csv
Rows: 200
Columns: 5


In [46]:
threshold = 60

customer_scores['Predicted_Churn'] = (
    customer_scores['Churn_Score'] >= threshold
).astype(int)

tp = ((customer_scores['Churned'] == 1) & 
      (customer_scores['Predicted_Churn'] == 1)).sum()

fp = ((customer_scores['Churned'] == 0) & 
      (customer_scores['Predicted_Churn'] == 1)).sum()

tn = ((customer_scores['Churned'] == 0) & 
      (customer_scores['Predicted_Churn'] == 0)).sum()

fn = ((customer_scores['Churned'] == 1) & 
      (customer_scores['Predicted_Churn'] == 0)).sum()

false_positive_rate = fp / (fp + tn)
recall_threshold = tp / (tp + fn)

print("Score Threshold:", threshold)
print("True Positives:", tp)
print("False Positives:", fp)
print("True Negatives:", tn)
print("False Negatives:", fn)
print("False Positive Rate:", round(false_positive_rate * 100, 2), "%")
print("Recall:", round(recall_threshold * 100, 2), "%")

if false_positive_rate < 0.25:
    print("\nAcceptance Criterion: PASS")
else:
    print("\nAcceptance Criterion: FAIL")

Score Threshold: 60
True Positives: 5
False Positives: 2
True Negatives: 192
False Negatives: 1
False Positive Rate: 1.03 %
Recall: 83.33 %

Acceptance Criterion: PASS


In [47]:
final_output = '../output/customer_churn_scores.csv'

customer_scores.to_csv(final_output, index=False)

print("Final score file saved successfully!")
print("Rows:", len(customer_scores))
print("Columns:", len(customer_scores.columns))
print("\nColumns:")
print(customer_scores.columns.tolist())

Final score file saved successfully!
Rows: 200
Columns: 6

Columns:
['Customer_ID', 'Churned', 'Churn_Probability', 'Churn_Score', 'Risk_Category', 'Predicted_Churn']
